In [0]:
%restart_python

In [0]:
%load_ext autoreload

In [0]:
%autoreload 2

In [0]:
from src.transforms.gold import build_dim_date, latest_actor_state, event_type_dim

CATALOG = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

events = spark.table(f"{CATALOG}.silver.events")

build_dim_date(spark, "2026-05-01", "2026-12-31").write.mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.dim_date")

latest_actor_state(events).write.mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.dim_actor")
event_type_dim(events).write.mode("overwrite")\
    .saveAsTable(f"{CATALOG}.gold.dim_event_type")
print("dim built")


In [0]:
spark.sql(f"SELECT count(*) AS days FROM {CATALOG}.gold.dim_date").show()

spark.sql(f"SELECT count(*) AS actors, sum(cast(is_bot AS int)) AS bots FROM {CATALOG}.gold.dim_actor").show()
spark.sql(f"SELECT * FROM {CATALOG}.gold.dim_event_type ORDER BY event_type").show(30)
spark.sql(f"""
          SELECT date_key, date, day_name, is_weekend
          FROM {CATALOG}.gold.dim_date WHERE date = '2026-06-01'
          """).show()

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.gold.dim_repo (
              repo_sk BIGINT GENERATED ALWAYS AS IDENTITY,
              repo_id BIGINT NOT NULL,
              repo_name STRING NOT NULL,
              owner STRING,
              language STRING,
              effective_from TIMESTAMP,
              effective_to TIMESTAMP,
              is_current BOOLEAN NOT NULL
          )""")

In [0]:
from src.transforms.gold import latest_repo_state
from src.writers.gold_writer import scd2_upsert_dim_repo
events = spark.table(f"{CATALOG}.silver.events")
scd2_upsert_dim_repo(latest_repo_state(events), CATALOG)

spark.sql(f"""
          SELECT count(*) total, sum(cast(is_current AS int)) current_rows
          FROM {CATALOG}.gold.dim_repo
          """).show()
          

In [0]:
spark.sql(f"SELECT count(*) FROM {CATALOG}.gold.dim_repo WHERE language IS NOT NULL").show()

In [0]:
from pyspark.sql import functions as F
from src.writers.gold_writer import scd2_upsert_dim_repo

def _drill(name):
    return (
        spark.createDataFrame(
    [(999999901, name, "drill-org", None)],
    "repo_id bigint, repo_name string, owner string, language string",
    )
    .withColumn("observed_at", F.current_timestamp())
    .withColumn("first_seen_ts", F.current_timestamp())
    .select("repo_id", "repo_name", "owner", "observed_at", "first_seen_ts", "language")
    )

scd2_upsert_dim_repo(_drill("drill-org/original-name"), CATALOG)
scd2_upsert_dim_repo(_drill("drill-org/renamed"), CATALOG)

spark.sql(f"""
          SELECT repo_sk, repo_name, effective_from, effective_to, is_current
          FROM {CATALOG}.gold.dim_repo WHERE repo_id = 999999901 ORDER BY effective_from
          """).show(truncate = False)

In [0]:
spark.sql(f"DELETE FROM {CATALOG}.gold.dim_repo WHERE repo_id = 999999901")